# Step 5 — Convert boundary to H3 cells

**H3** is Uber's hierarchical hexagonal spatial indexing system.
This notebook converts your boundary polygon into a set of H3 cell IDs
that cover the area at a chosen resolution.

## Why H3 cells?

The duckOSM road network already has H3 indices on every edge (`from_cell`, `to_cell`).
Generating the boundary's H3 cells lets you:
- Partition routing queries by cell (faster lookups)
- Join the road network with other H3-indexed datasets (traffic counts, population, etc.)
- Visualise coverage in tools like **kepler.gl** (which renders H3 IDs natively)

## Resolution guide

| Resolution | Approx cell area | Cells for ~10 km² district |
|---|---|---|
| 7 | ~5 km² | ~2 |
| 8 | ~0.7 km² | ~15 |
| 9 | ~0.1 km² | ~100 |
| 10 | ~0.015 km² | ~650 |
| 15 | microscopic | millions |

Resolution **8** is a good default for district-level analysis.  
Match the resolution to the one used in duckOSM (`h3_resolution` in `config/*.yaml`).

In [21]:
%pip install h3 folium --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%%time
import json
import yaml
import h3
import duckdb
import folium
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import shape, mapping

# ── Configuration ─────────────────────────────────────────────────────────
NAME       = 'sodermalm'
BOUNDARY_FILE = Path(f'../boundaries/{NAME}.geojson')
OUTPUT_DIR = Path('../output')
# ─────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(exist_ok=True)

# Read resolutions and DB path from config YAML
CONFIG_PATH = Path(f'../config/{NAME}.yaml')
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        cfg = yaml.safe_load(f)
    DB_PATH     = Path(cfg.get('output_path', '../db')) / f"{cfg['name']}.duckdb"
    RESOLUTIONS = cfg.get('h3_resolutions', [8])
else:
    DB_PATH     = Path(f'../db/{NAME}.duckdb')
    RESOLUTIONS = [8]
    print('WARNING: config not found — using default DB path and resolution 8')

print(f'Boundary    : {BOUNDARY_FILE}')
print(f'Resolutions : {RESOLUTIONS}')
print(f'DuckDB      : {DB_PATH}')
for r in RESOLUTIONS:
    edge_km = h3.average_hexagon_edge_length(r, 'km')
    print(f'  Resolution {r}: cell area ~{h3.average_hexagon_area(r, "km^2"):.2f} km²  '
          f'edge ~{edge_km:.3f} km')

---
## Convert polygon to H3 cells

We slightly **buffer** the boundary before converting.
Without a buffer, cells that touch the boundary edge are sometimes excluded,
leaving gaps around the perimeter.

`h3.polygon_to_cells()` requires coordinates in **(lat, lon)** order — the opposite
of GeoJSON which uses **(lon, lat)**. We swap them during conversion.

In [ ]:
%%time
# Load boundary
with open(BOUNDARY_FILE) as f:
    geojson_data = json.load(f)

feature  = geojson_data['features'][0]
geometry = feature['geometry']
shapely_poly = shape(geometry)

# h3-py v4 expects (lat, lon) — swap from GeoJSON (lon, lat)
def swap_coords(coords):
    return [(lat, lon) for lon, lat in coords]

# Compute cells for each resolution
cells_by_res = {}
for res in RESOLUTIONS:
    # Buffer by half the hex edge length so cells fully cover the boundary edge
    buf_deg = h3.average_hexagon_edge_length(res, 'km') / 111.0 * 0.5
    buffered = mapping(shapely_poly.buffer(buf_deg))
    outer    = swap_coords(buffered['coordinates'][0])
    holes    = [swap_coords(r) for r in buffered['coordinates'][1:]]
    poly     = h3.LatLngPoly(outer, *holes)
    cells_by_res[res] = list(h3.polygon_to_cells(poly, res))
    print(f'Resolution {res}: {len(cells_by_res[res])} cells')

---
## Visualize

The boundary is shown in red, H3 cells in blue.  
Check that the cells fully cover the boundary without large gaps at the edges.

In [ ]:
%%time
# Visualise the finest resolution (last in RESOLUTIONS list)
vis_res = RESOLUTIONS[-1]
cells   = cells_by_res[vis_res]

coords = geometry['coordinates'][0]
center = [sum(c[1] for c in coords) / len(coords),
          sum(c[0] for c in coords) / len(coords)]

m = folium.Map(location=center, zoom_start=13, tiles='OpenStreetMap')

folium.GeoJson(
    geojson_data,
    style_function=lambda _: {'color': 'red', 'weight': 2, 'fillOpacity': 0},
    tooltip='Boundary',
).add_to(m)

for cell in cells:
    folium.Polygon(
        locations=h3.cell_to_boundary(cell),
        color='blue', weight=1,
        fill=True, fill_color='blue', fill_opacity=0.1,
        tooltip=cell,
    ).add_to(m)

print(f'Showing resolution {vis_res} ({len(cells)} cells)')
m

---
## Save

For each resolution, saves:

| File | Use with |
|---|---|
| `{name}_{resolution}_h3_cells.csv` | **kepler.gl** — drag and drop, hexagons render automatically |
| `{name}_{resolution}_h3_cells.geojson` | QGIS, geopandas, any GIS tool |

All resolutions are written to the **same** `boundary_cells` table in DuckDB
(distinguished by the `resolution` column). Re-running replaces only the rows
for each resolution without touching the others.

In [ ]:
%%time
import pandas as pd

con = duckdb.connect(str(DB_PATH))
con.execute('LOAD spatial')

con.execute("""
    CREATE TABLE IF NOT EXISTS boundary_cells (
        h3_id      VARCHAR PRIMARY KEY,
        resolution INTEGER,
        geometry   GEOMETRY
    )
""")

for RESOLUTION, cells in cells_by_res.items():
    # ── CSV ───────────────────────────────────────────────────────────────
    csv_path = OUTPUT_DIR / f'{NAME}_{RESOLUTION}_h3_cells.csv'
    pd.DataFrame({'h3_id': cells}).to_csv(csv_path, index=False)

    # ── GeoJSON ───────────────────────────────────────────────────────────
    cell_polygons = []
    for cell in cells:
        coords = [(lon, lat) for lat, lon in h3.cell_to_boundary(cell)]
        coords.append(coords[0])
        cell_polygons.append({
            'type': 'Feature',
            'geometry': {'type': 'Polygon', 'coordinates': [coords]},
            'properties': {'h3_id': cell, 'resolution': RESOLUTION},
        })
    geojson_path = OUTPUT_DIR / f'{NAME}_{RESOLUTION}_h3_cells.geojson'
    with open(geojson_path, 'w') as f:
        json.dump({'type': 'FeatureCollection', 'features': cell_polygons}, f)

    # ── DuckDB — replace only this resolution's rows ───────────────────────
    con.execute(f"DELETE FROM boundary_cells WHERE resolution = {RESOLUTION}")
    cells_df = pd.DataFrame([
        {
            'h3_id':        cell,
            'resolution':   RESOLUTION,
            'geometry_wkt': 'POLYGON ((' +
                             ', '.join(f'{lon} {lat}'
                                       for lon, lat in feat['geometry']['coordinates'][0][:-1]) +
                             ', ' + '{} {}'.format(*feat['geometry']['coordinates'][0][0]) + '))',
        }
        for cell, feat in zip(cells, cell_polygons)
    ])
    con.register('_cells_temp', cells_df)
    con.execute("INSERT INTO boundary_cells SELECT h3_id, resolution, ST_GeomFromText(geometry_wkt) FROM _cells_temp")
    con.unregister('_cells_temp')

    print(f'Resolution {RESOLUTION}: {len(cells)} cells → CSV, GeoJSON, DuckDB')

summary = con.execute(
    "SELECT resolution, count(*) AS cells FROM boundary_cells GROUP BY resolution ORDER BY resolution"
).df()
con.close()

print('\nAll resolutions stored in boundary_cells:')
display(summary)